# Libraries

In [3]:
!pip -q install "lm_eval[hf]"
!pip -q install compressed-tensors

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.4/56.4 kB 3.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 101.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.1/91.1 kB 7.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.5/196.5 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 107.2 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 28.1 MB/s eta 0:00:00


In [4]:
# 2. Libraries
import random
import numpy as np
import torch
import os
import gc

import lm_eval
import subprocess

from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

import shutil
from IPython.display import FileLink

# Functions

In [5]:
# --- 2. REPRODUCIBILITY ---
def set_reproducibility(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # Ensure deterministic behavior in CuDNN
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"Global seed set to {seed}")

# Global Variables

In [6]:
model_name = "Qwen/Qwen3-4B"
SEED = 42
os.environ["HF_ALLOW_CODE_EVAL"] = "1"

# Seed for reproducibility

In [7]:
set_reproducibility(SEED)

Global seed set to 42


# Login to huggingface

In [8]:
# --- 3. HUGGING FACE AUTHENTICATION ---
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)
print("Logging into Hugging Face...")

Logging into Hugging Face...


# Do Evaluation

**Configurations**

In [9]:
tasks_config = {
    "gsm8k": {
        "task_name": "gsm8k_cot",
        "fewshot": 2   # 🔻 reduced from 8
    },
    "mmlu": {
        "task_name": "mmlu",
        "fewshot": 2   # 🔻 reduced from 5
    },
    "arc_challenge": {
        "task_name": "arc_challenge",
        "fewshot": 0   # keep
    },
    "wikitext": {
        "task_name": "wikitext",
        "fewshot": 0   # keep
    }
}

In [10]:
for task, cfg in tasks_config.items():
    print(f"Starting evaluation for: {task}")

    command = [
        "lm_eval",
        "--model", "hf",
        "--model_args", f"pretrained={model_name},max_length=2048,dtype=float16,parallelize=True",
        "--tasks", cfg["task_name"],
        "--batch_size", str(2),   
        "--verbosity", "WARNING",
        "--seed", str(SEED),
        "--limit", str(100),
        "--num_fewshot", str(cfg["fewshot"]),
        "--output_path", f"evaluation_results/{task}"
    ]

    subprocess.run(command, check=True)
    
    torch.cuda.empty_cache()
    gc.collect()
    print(f"Finished {task}\n")

Starting evaluation for: gsm8k


2026-03-26:14:13:28 WARNING  [config.evaluate_config:281] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-03-26:14:13:34 INFO     [_cli.run:376] Selected Tasks: ['gsm8k_cot']
2026-03-26 14:13:48.486833: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774534428.694956     128 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774534428.751265     128 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774534429.271327     128 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774534429.271364     128 co

hf ({'pretrained': 'Qwen/Qwen3-4B', 'max_length': 2048, 'dtype': 'float16', 'parallelize': True}), gen_kwargs: ({}), limit: 100.0, num_fewshot: 2, batch_size: 2
|  Tasks  |Version|     Filter     |n-shot|  Metric   |   |Value|   |Stderr|
|---------|------:|----------------|-----:|-----------|---|----:|---|-----:|
|gsm8k_cot|      3|flexible-extract|     2|exact_match|↑  | 0.84|±  |0.0368|
|         |       |strict-match    |     2|exact_match|↑  | 0.82|±  |0.0386|

Finished gsm8k

Starting evaluation for: mmlu


2026-03-26:14:22:50 WARNING  [config.evaluate_config:281] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-03-26:14:22:55 INFO     [_cli.run:376] Selected Tasks: ['mmlu']
2026-03-26 14:23:02.321085: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774534982.339667     729 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774534982.345855     729 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774534982.361833     729 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774534982.361856     729 computa

hf ({'pretrained': 'Qwen/Qwen3-4B', 'max_length': 2048, 'dtype': 'float16', 'parallelize': True}), gen_kwargs: ({}), limit: 100.0, num_fewshot: 2, batch_size: 2
|                 Tasks                 |Version|Filter|n-shot|Metric|   |Value |   |Stderr|
|---------------------------------------|------:|------|-----:|------|---|-----:|---|-----:|
|mmlu                                   |      2|none  |      |acc   |↑  |0.7149|±  |0.0058|
| - humanities                          |      2|none  |      |acc   |↑  |0.7069|±  |0.0120|
|  - formal_logic                       |      1|none  |     2|acc   |↑  |0.6200|±  |0.0488|
|  - high_school_european_history       |      1|none  |     2|acc   |↑  |0.7500|±  |0.0435|
|  - high_school_us_history             |      1|none  |     2|acc   |↑  |0.8700|±  |0.0338|
|  - high_school_world_history          |      1|none  |     2|acc   |↑  |0.8400|±  |0.0368|
|  - international_law                  |      1|none  |     2|acc   |↑  |0.7600|±  |0.0429|
| 

2026-03-26:14:48:29 WARNING  [config.evaluate_config:281] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-03-26:14:48:34 INFO     [_cli.run:376] Selected Tasks: ['arc_challenge']
2026-03-26 14:48:42.155300: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774536522.173405    1186 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774536522.179268    1186 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774536522.195998    1186 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774536522.196019    118

hf ({'pretrained': 'Qwen/Qwen3-4B', 'max_length': 2048, 'dtype': 'float16', 'parallelize': True}), gen_kwargs: ({}), limit: 100.0, num_fewshot: 0, batch_size: 2
|    Tasks    |Version|Filter|n-shot| Metric |   |Value|   |Stderr|
|-------------|------:|------|-----:|--------|---|----:|---|-----:|
|arc_challenge|      1|none  |     0|acc     |↑  | 0.53|±  |0.0502|
|             |       |none  |     0|acc_norm|↑  | 0.51|±  |0.0502|

Finished arc_challenge

Starting evaluation for: wikitext


2026-03-26:14:49:20 WARNING  [config.evaluate_config:281] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-03-26:14:49:25 INFO     [_cli.run:376] Selected Tasks: ['wikitext']
2026-03-26 14:49:32.664700: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774536572.683229    1264 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774536572.689316    1264 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774536572.705338    1264 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774536572.705359    1264 com

hf ({'pretrained': 'Qwen/Qwen3-4B', 'max_length': 2048, 'dtype': 'float16', 'parallelize': True}), gen_kwargs: ({}), limit: 100.0, num_fewshot: 0, batch_size: 2
| Tasks  |Version|Filter|n-shot|    Metric     |   | Value |   |Stderr|
|--------|------:|------|-----:|---------------|---|------:|---|------|
|wikitext|      2|none  |     0|bits_per_byte  |↓  | 0.7860|±  |   N/A|
|        |       |none  |     0|byte_perplexity|↓  | 1.7243|±  |   N/A|
|        |       |none  |     0|word_perplexity|↓  |18.4202|±  |   N/A|

Finished wikitext



# Save reports

In [15]:
import shutil
from IPython.display import FileLink

zip_path = shutil.make_archive(
    "/kaggle/working/evaluation_results",  # full path
    'zip',
    "/kaggle/working/evaluation_results"   # folder to zip
)

FileLink(zip_path)

/kaggle/working/evaluation_results.zip

In [17]:
zip_path = shutil.make_archive(f"evaluation_results", 'zip', f"evaluation_results")
zip_path


'/kaggle/working/evaluation_results.zip'